# case-study-1116 — Case study: how sure is the manager? Which factors do we keep?

**ML & FinTech · 115-1 · 20261116 · Session 3 · pairs drawn by card · ONE laptop per pair · 40 minutes, then discussion**

| Part | Min | What you do |
|---|---|---|
| Case 1 | 14 | A drives, B navigates, desk A. The bootstrap and the splits: how sure is the manager? |
| Case 2 | 14 | Swap: B drives, A navigates, desk B. Forty candidate factors: is the R² real? |
| Hand in | 8 | A pair verdict on **one sheet of paper per pair** |

You may use any AI tool for the code. Hand in the paper sheet and you get full credit for today's case study; be ready to explain your numbers. Your numbers come from your 學號; your partner's differ.

**Your desk** = your own data: your 學號 draws 10 of 50 US stocks (daily returns in 2017, in %). Desk A uses A's 學號, desk B uses B's.

**Roles.** The *Driver* types and asks the AI. The *Navigator* holds the paper sheet, works the hand step, checks every number and says when it is done. Swap at Case 2.


## Setup

In [ ]:
import hashlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import warnings; warnings.filterwarnings("ignore")

In [ ]:
STUDENT_ID = ""    # A: 學號 of the Driver in Case 1, in quotes, e.g. "315707999"
PARTNER_ID = ""    # B: 學號 of the Driver in Case 2

In [ ]:
URL  = ("https://raw.githubusercontent.com/202609-ML-FinTech/00-course-info/"
        "main/in-class-exercise/historical_stock_prices_2017.csv")
CAND = ["historical_stock_prices_2017.csv", "../historical_stock_prices_2017.csv",
        "../00-course-info/in-class-exercise/historical_stock_prices_2017.csv"]
path = next((p for p in CAND if Path(p).exists()), URL)
prices = pd.read_csv(path, index_col=0).dropna(axis=1); prices.index = pd.to_datetime(prices.index)
ret = np.log(prices).diff().dropna() * 100                # daily log returns, %

def draw_portfolio(student_id, k=10):
    key = lambda t: hashlib.sha256(f"{int(student_id)}-{t}".encode()).hexdigest()
    return sorted(sorted(prices.columns, key=key)[:k])

assert str(STUDENT_ID).strip().isdigit() and str(PARTNER_ID).strip().isdigit(), "Fill in both IDs in the cell above, then run again."
DESK = {}
for who, sid in (("A", STUDENT_ID), ("B", PARTNER_ID)):
    P = draw_portfolio(sid); a, b = P[0], P[1]; x_, z_ = ret[a].values, ret[b].values
    DESK[who] = dict(id=sid, portfolio=P, A=a, B=b)
    print(f"Desk {who} ({sid}): stock A = {a}, stock B = {b}   var(A) = {x_.var(ddof=1):.3f}   var(B) = {z_.var(ddof=1):.3f}   cov(A,B) = {np.cov(x_, z_)[0,1]:.3f}")
print("Navigator: compute alpha for desk A by calculator from these three numbers.")
MY_PORTFOLIO = DESK["A"]["portfolio"]; A, B = DESK["A"]["A"], DESK["A"]["B"]
x, z = ret[A].values, ret[B].values


---
# Case 1 — How sure is the manager? (A drives, B navigates, desk A; 14 min)

Stock A and stock B are the first two of your 10 stocks. A portfolio puts weight α on A and 1 − α on B; its variance is smallest at α = (σ²_B − σ_AB)/(σ²_A + σ²_B − 2σ_AB) (ISLP §5.2). The bootstrap redraws the days with replacement 1,000 times; SE = standard deviation of the 1,000 values of α; the 95% range runs from their 2.5% point to their 97.5% point.

The manager says: *"The best weight on stock A is exactly the number I computed, to one decimal. And my model has 61% accuracy."*

**(a)** Navigator: compute α for desk A by calculator from the printed numbers. Driver: compute α = (σ²_B − σ_AB)/(σ²_A + σ²_B − 2σ_AB). Draw 1,000 bootstrap samples of the days (keep A and B paired; `rng = np.random.default_rng(int(STUDENT_ID))`). Do it for **both desks** (seed `int(<that desk's ID>)`). Print α, its standard error and the 2.5%–97.5% range. The Navigator's hand α for desk A must match.


In [ ]:
# 1a: alpha; bootstrap SE; 95% range, for desk A and desk B (DESK['A'], DESK['B'])




**(c)** The 61%: a logistic regression predicts whether tomorrow's return on stock A is positive, from today's 5-day mean and 5-day standard deviation (features given). A random split puts later days in the training set, so the model learns from the future; a time-ordered split trains on the first 150 days and tests on the rest, as in real trading. On desk A, estimate accuracy: three random 60/40 splits (`random_state=1, 2, 3`), 5-fold CV (`KFold(5, shuffle=True, random_state=0)`), and a time-ordered split (fit on the first 150 days, test on the rest).


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, KFold, cross_val_score
s = ret[A]
F = pd.DataFrame({"m5": s.rolling(5).mean(), "v5": s.rolling(5).std()})
y = (s.shift(-1) > 0).astype(int)                 # tomorrow up?
ok = F.notna().all(axis=1) & s.shift(-1).notna()
F, y = F[ok], y[ok]
# 1c: three random splits, 5-fold CV, the time-ordered split




**(a)** Desk A: α ____ ; SE ____ ; range ____ to ____ . Desk B: α ____ ; SE ____ ; range ____ to ____ . **(c)** Accuracy: splits ____ , ____ , ____ ; CV ____ ; time-ordered ____ .

**(b)** Whose α is more certain, desk A or desk B? Does either 95% range include 0.5 (an equal split)? Is one decimal justified? **(c)** Is 61% honest?

> _replace this line_

**(d)** The client wants one weight for stock A and one accuracy figure. Agree between you: what do you write, and with how many decimals?

> _replace this line_


---
# Case 2 — Which of 40 factors do we keep? (swap: B drives, A navigates; 14 min)

Target: the daily return of stock A on **desk B**. Candidates: 40 factors, F01 to F40: 20 are other stocks' returns and 20 are pure random noise (you are not told which). *Honest R²* = the R² on days the model never saw, with the five factors chosen without looking at those days.

The analyst says: *"I ranked all 40 by correlation with the target and kept the best five. The R² is high, and even cross-validation confirms it."*

**(a) Before you run anything**, each of you writes a guess on the sheet: the honest R² of the five-factor model.


In [ ]:
MY_PORTFOLIO = DESK["B"]["portfolio"]; TARGET = DESK["B"]["A"]
yr = ret[TARGET]
others = [t for t in sorted(prices.columns) if t not in MY_PORTFOLIO][:20]
rng = np.random.default_rng(int(hashlib.sha256(f"{int(PARTNER_ID)}-noise".encode()).hexdigest(), 16) % 2**32)
X = ret[others].copy()
for j in range(20):
    X[f"noise{j}"] = rng.normal(0, 1, len(X))
X = X.iloc[:, rng.permutation(40)]                # shuffled, then anonymised
X.columns = [f"F{j+1:02d}" for j in range(40)]
print(f"Target: {TARGET}   candidates: {X.shape[1]} factors, {len(X)} days")


**(b)** Pick the five factors with the largest |correlation| with `yr` on the full sample. Print: (i) in-sample R²; (ii) mean test R² with `KFold(5, shuffle=True, random_state=1)` keeping those five fixed (chosen once on all days); (iii) mean test R² when the five are chosen **inside** each fold, on the training part only. Navigator: for (ii) and (iii), did the test days help choose the five? (iv) Then fit `LassoCV(cv=5, random_state=0)` on all 40 standardised factors and count the non-zero coefficients.


In [ ]:
# 2: in-sample R2; CV with fixed factors; CV with selection inside; lasso




**(b)** In-sample R² ____ ; CV fixed ____ ; CV inside ____ ; the lasso keeps ____ factors, ____ of the five.

**(c)** Is the analyst right? Which number is the honest one? Whose guess was closer?

> _replace this line_

**(c, continued)** By how much does the in-sample R² overstate your honest R² (selection inside the folds)? Would you pay for this five-factor model?

> _replace this line_


---
# Hand in, on paper (10 min)

**One sheet per pair**: both names and 學號, the numbers of both desks, and the **pair verdict**: one agreed sentence per case, where you disagreed and how you settled it, and one thing your partner saw that you missed.

Give the sheet to Prof. Teng or Claire before you leave.

Then: *Restart & Run All*, and **each of you** pushes a copy of this notebook as `in-class-exercise/1116/case-study-1116.ipynb` in your own repository before **12:10**.
